# 99 — Reset (presenter only)
Removes what **this PoC** created (demo objects, PoC tables and their data buckets), restores the throttler backup, re-seeds, and starts a new run ID. Re-runnable.

## Step 1 — Dry run: what would be deleted
Nothing is changed in this step. Review the list — buckets ending in `--table-s3` that do not belong to a PoC table are listed as *left alone* and are never deleted.

In [ ]:
import pandas as pd
from poc import results, viz
from poc.reset import full_reset

def show(r):
    viz.table(pd.DataFrame([r["objects_removed"]], index=["demo-* objects"]))
    viz.table(pd.DataFrame({"tables": r["tables_removed"] or ["(none)"]}))
    viz.table(pd.DataFrame(
        [{"data bucket": b, "state": s} for b, s in r["data_buckets_deleted"].items()]
        + [{"data bucket": b, "state": "left alone (not a PoC table)"}
           for b in r["unattributed_left_alone"]]
        or [{"data bucket": "(none)", "state": ""}]))
    for note in r["notes"]:
        print("note:", note)

print("previous run id:", results.run_id())
plan = full_reset(dry_run=True)
show(plan)
print("throttler:", plan["restore_throttler"], "| table bucket:", plan["table_bucket_deleted"])

## Step 2 — Full reset
Restores `reports/throttle_backup.json`, deletes `demo-*` objects in both buckets, drops the PoC tables/namespaces, their own data buckets and the PoC table bucket, then re-seeds bucket + namespace and starts a new run ID.

In [ ]:
done = full_reset()
show(done)
problems = [f"{b}: {s}" for b, s in done["data_buckets_deleted"].items() if s.startswith("FAILED")]
problems += [t for t in done["tables_removed"] if "FAILED" in t]
pending = [b for b, s in done["data_buckets_deleted"].items() if s.startswith("pending")]
RID = results.new_run()
print("throttler restored:", done["restore_throttler"])
print("seed:", done["seed"], "| new run id:", RID)
viz.banner(not problems, "reset",
           f"new run {RID}" + (f"; {len(pending)} data bucket(s) still being emptied "
                               f"by ObjectScale — retried on the next reset" if pending else "")
           + (f"; problems: {problems}" if problems else ""))

**Presenter notes** (hidden)

ObjectScale empties a dropped table's data bucket by itself and refuses changes meanwhile (EmptyBucketInProgress) — such buckets are reported as pending and remembered for the next reset.